# The Bark Bias Cure: Dog Behavior Analysis

This notebook demonstrates how to analyze bias in AI systems using a fun, synthetic example of dog behavior classification. While the scenario is fictional, the underlying principles of bias detection, measurement, and mitigation are real and applicable to serious AI applications.

## Project Overview

The goal of this project is to:
1. **Create awareness** about bias in AI systems
2. **Demonstrate** how bias can manifest in machine learning models
3. **Provide tools** for detecting and measuring bias
4. **Show techniques** for mitigating bias in AI systems

## Table of Contents

1. [Data Generation](#data-generation)
2. [Exploratory Data Analysis](#exploratory-data-analysis)
3. [Model Training](#model-training)
4. [Bias Analysis](#bias-analysis)
5. [Fairness Metrics](#fairness-metrics)
6. [Bias Mitigation](#bias-mitigation)
7. [Conclusions](#conclusions)


In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

print("📊 Libraries imported successfully!")
print("🐕 Welcome to The Bark Bias Cure project!")


ModuleNotFoundError: No module named 'pandas'

## 1. Data Generation

First, let's generate our synthetic dataset that demonstrates bias patterns in dog behavior. This dataset will show how dogs might behave differently towards people based on various attributes.


In [ ]:
# Import our data generation utility
import sys
sys.path.append('../utils')
from generate_data import DogBehaviorGenerator

# Generate the dataset
print("🐕 Generating synthetic dog behavior dataset...")
generator = DogBehaviorGenerator(seed=42)

# Create dataset with bias patterns
dataset = generator.generate_dataset(
    n_humans=1000,
    n_dogs=200, 
    n_interactions=5000
)

# Display basic information
print(f"\n📊 Dataset Summary:")
print(f"Total humans: {len(dataset['humans'])}")
print(f"Total dogs: {len(dataset['dogs'])}")
print(f"Total interactions: {len(dataset['interactions'])}")
print(f"Barking incidents: {dataset['interactions']['barks'].sum()}")
print(f"Overall barking rate: {dataset['interactions']['barks'].mean():.2%}")

# Show first few interactions
print(f"\n🔍 Sample interactions:")
print(dataset['interactions'][['human_race', 'human_gender', 'human_age_group', 'barks', 'bias_score']].head(10))


## 2. Exploratory Data Analysis

Let's explore the dataset to understand the bias patterns and relationships between different attributes.


In [ ]:
# Analyze bias by race
print("🔍 Bias Analysis by Race:")
race_analysis = dataset['interactions'].groupby('human_race')['barks'].agg(['count', 'sum', 'mean'])
race_analysis = race_analysis.sort_values('mean', ascending=False)

for race in race_analysis.index:
    rate = race_analysis.loc[race, 'mean']
    count = race_analysis.loc[race, 'count']
    print(f"  {race:<15}: {rate:.2%} ({race_analysis.loc[race, 'sum']}/{count})")

# Visualize bias by race
plt.figure(figsize=(12, 6))
race_bias = dataset['interactions'].groupby('human_race')['barks'].mean().sort_values(ascending=False)
bars = plt.bar(race_bias.index, race_bias.values, color=plt.cm.Reds(np.linspace(0.3, 1, len(race_bias))))
plt.title('Barking Rate by Human Race\n(Showing bias in the dataset)', fontsize=14, fontweight='bold')
plt.xlabel('Race')
plt.ylabel('Barking Rate')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Add value labels
for bar, value in zip(bars, race_bias.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{value:.2%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Analyze bias by gender
print("\n🔍 Bias Analysis by Gender:")
gender_analysis = dataset['interactions'].groupby('human_gender')['barks'].agg(['count', 'sum', 'mean'])
gender_analysis = gender_analysis.sort_values('mean', ascending=False)

for gender in gender_analysis.index:
    rate = gender_analysis.loc[gender, 'mean']
    count = gender_analysis.loc[gender, 'count']
    print(f"  {gender:<15}: {rate:.2%} ({gender_analysis.loc[gender, 'sum']}/{count})")

# Visualize bias by gender
plt.figure(figsize=(10, 6))
gender_bias = dataset['interactions'].groupby('human_gender')['barks'].mean().sort_values(ascending=False)
bars = plt.bar(gender_bias.index, gender_bias.values, color=plt.cm.Blues(np.linspace(0.3, 1, len(gender_bias))))
plt.title('Barking Rate by Human Gender\n(Showing bias in the dataset)', fontsize=14, fontweight='bold')
plt.xlabel('Gender')
plt.ylabel('Barking Rate')
plt.grid(True, alpha=0.3)

# Add value labels
for bar, value in zip(bars, gender_bias.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{value:.2%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 3. Model Training

Now let's train an XGBoost model to predict dog behavior. We'll use this model to demonstrate how bias can be perpetuated and amplified by machine learning systems.


In [ ]:
# Import our training utility
from train_model import DogBehaviorClassifier

# Prepare the data
df = dataset['interactions'].copy()
print(f"📊 Dataset shape: {df.shape}")

# Initialize classifier
classifier = DogBehaviorClassifier(random_state=42)

# Prepare features
print("🔧 Preparing features...")
X = classifier.prepare_features(df)
y = df['barks']

print(f"Features shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Train model
print("\n🚀 Training XGBoost model...")
results = classifier.train(X, y, test_size=0.2, optimize_hyperparams=False)  # Set to True for hyperparameter optimization

print(f"\n✅ Model training completed!")
print(f"AUC Score: {results['auc_score']:.4f}")
print(f"Accuracy: {results['accuracy']:.4f}")


In [ ]:
# Analyze feature importance
print("🔍 Top 10 Most Important Features:")
feature_importance = results['feature_importance'].head(10)

plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(feature_importance)), feature_importance['importance'], 
                color=plt.cm.viridis(np.linspace(0, 1, len(feature_importance))))
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features for Dog Barking Prediction', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add value labels
for i, (bar, value) in enumerate(zip(bars, feature_importance['importance'])):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2, 
             f'{value:.3f}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Show feature importance table
print("\nFeature Importance Table:")
print(feature_importance[['feature', 'importance']].to_string(index=False))


## 4. Bias Analysis

Now let's analyze how the trained model performs across different groups to identify bias patterns.


In [ ]:
# Import evaluation utility
from evaluate_model import ModelEvaluator

# Create a temporary model file for evaluation
import joblib
import os
os.makedirs('../models', exist_ok=True)

# Save the trained model
model_data = {
    'model': classifier.model,
    'scaler': classifier.scaler,
    'label_encoders': classifier.label_encoders,
    'feature_columns': classifier.feature_columns
}
joblib.dump(model_data, '../models/dog_behavior_classifier.joblib')

# Initialize evaluator
evaluator = ModelEvaluator('../models/dog_behavior_classifier.joblib')
evaluator.load_model()

# Get predictions for bias analysis
y_pred, y_proba = evaluator.predict(df)

# Analyze bias by race
print("🔍 Model Bias Analysis by Race:")
race_bias_analysis = {}

for race in df['human_race'].unique():
    race_mask = df['human_race'] == race
    if race_mask.sum() > 10:  # Only analyze if enough samples
        race_df = df[race_mask]
        race_y_true = race_df['barks'].values
        race_y_pred = y_pred[race_mask]
        race_y_proba = y_proba[race_mask]
        
        actual_rate = race_y_true.mean()
        predicted_rate = race_y_pred.mean()
        bias_score = predicted_rate - actual_rate
        
        race_bias_analysis[race] = {
            'actual_rate': actual_rate,
            'predicted_rate': predicted_rate,
            'bias_score': bias_score,
            'n_samples': len(race_df)
        }
        
        print(f"  {race:<15}: Actual={actual_rate:.2%}, Predicted={predicted_rate:.2%}, Bias={bias_score:+.2%}")

# Visualize model bias by race
plt.figure(figsize=(12, 6))
races = list(race_bias_analysis.keys())
bias_scores = [race_bias_analysis[race]['bias_score'] for race in races]

bars = plt.bar(races, bias_scores, color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(races))))
plt.title('Model Bias by Race\n(Predicted Rate - Actual Rate)', fontsize=14, fontweight='bold')
plt.xlabel('Race')
plt.ylabel('Bias Score')
plt.xticks(rotation=45)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
plt.grid(True, alpha=0.3)

# Add value labels
for bar, score in zip(bars, bias_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.01 if score >= 0 else -0.02), 
             f'{score:+.2%}', ha='center', va='bottom' if score >= 0 else 'top')

plt.tight_layout()
plt.show()
